# Finance Projected Growth Analysis

This notebook rebuilds the project around:
- a small regression model that predicts `projected growth %` from AI/O*NET occupation features
- a `two-sample Welch t-test` comparing `judgment-intensive` and `process-driven` finance occupations

The projected-growth values come from the official O*NET local trends pages and are cached locally after the first fetch.


In [ ]:
%pip install pandas matplotlib openpyxl

In [ ]:
import io
import math
import re
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('/Users/stephenye/stephen/coding/project/datajam')
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
REPORTS_DIR = ROOT / 'reports'
AIOE_XLSX = RAW_DIR / 'aioe_data.xlsx'
ONET_ZIP = RAW_DIR / 'onet_30_2_excel.zip'
LOCALTRENDS_CACHE = PROCESSED_DIR / 'localtrends_metrics.csv'

FINANCE_GROUPS = {
    '13-2011': 'judgment-intensive',
    '13-2031': 'judgment-intensive',
    '13-2041': 'judgment-intensive',
    '13-2051': 'judgment-intensive',
    '13-2052': 'judgment-intensive',
    '13-2061': 'judgment-intensive',
    '13-2071': 'judgment-intensive',
    '13-1031': 'process-driven',
    '13-2053': 'process-driven',
    '13-2072': 'process-driven',
    '13-2081': 'process-driven',
    '13-2082': 'process-driven',
}

FEATURE_COLS = [
    'ai_exposure',
    'job_zone',
    'judgment_skills',
    'math_skill',
    'finance_knowledge',
    'process_activities',
    'advice_analysis',
]


In [ ]:
def normalize_soc(series):
    return series.astype(str).str.extract(r'(\d{2}-\d{4})', expand=False)


def read_onet_excel_from_zip(zip_file, file_name, **kwargs):
    with zip_file.open(f'db_30_2_excel/{file_name}') as handle:
        return pd.read_excel(io.BytesIO(handle.read()), **kwargs)


def aggregate_scale_average(table, element_names, scale_id, column_name):
    filtered = table[
        table['Element Name'].isin(element_names)
        & (table['Scale ID'] == scale_id)
    ].copy()
    return filtered.groupby('soc_code')['Data Value'].mean().rename(column_name)


def fit_linear_regression(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    X_design = np.column_stack([np.ones(len(X)), X])
    beta, *_ = np.linalg.lstsq(X_design, y, rcond=None)
    return beta


def predict_linear_regression(beta, X):
    X = np.asarray(X, dtype=float)
    X_design = np.column_stack([np.ones(len(X)), X])
    return X_design @ beta


def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = ((y_true - y_pred) ** 2).sum()
    ss_tot = ((y_true - y_true.mean()) ** 2).sum()
    return 1 - ss_res / ss_tot


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.abs(y_true - y_pred).mean()


def t_pdf(x, df):
    numerator = math.gamma((df + 1) / 2)
    denominator = math.sqrt(df * math.pi) * math.gamma(df / 2)
    return numerator / denominator * (1 + (x * x) / df) ** (-(df + 1) / 2)


def t_cdf_numeric(t_value, df, upper=80, steps=200000):
    if t_value == 0:
        return 0.5
    if t_value < 0:
        return 1 - t_cdf_numeric(-t_value, df, upper=upper, steps=steps)
    xs = np.linspace(0, min(float(t_value), upper), steps)
    ys = np.array([t_pdf(float(x), float(df)) for x in xs])
    area = np.trapezoid(ys, xs)
    return min(1.0, 0.5 + area)


def welch_t_test(sample_1, sample_2, alternative='greater'):
    sample_1 = np.asarray(sample_1, dtype=float)
    sample_2 = np.asarray(sample_2, dtype=float)
    n1 = len(sample_1)
    n2 = len(sample_2)
    mean_1 = sample_1.mean()
    mean_2 = sample_2.mean()
    var_1 = sample_1.var(ddof=1)
    var_2 = sample_2.var(ddof=1)
    standard_error = math.sqrt(var_1 / n1 + var_2 / n2)
    t_statistic = (mean_1 - mean_2) / standard_error
    degrees_of_freedom = ((var_1 / n1 + var_2 / n2) ** 2) / (
        ((var_1 / n1) ** 2) / (n1 - 1) + ((var_2 / n2) ** 2) / (n2 - 1)
    )
    cdf = t_cdf_numeric(t_statistic, degrees_of_freedom)
    p_value = 1 - cdf if alternative == 'greater' else cdf
    return {
        'n_1': n1,
        'n_2': n2,
        'mean_1': mean_1,
        'mean_2': mean_2,
        't_statistic': t_statistic,
        'degrees_of_freedom': degrees_of_freedom,
        'p_value': p_value,
    }


def fetch_localtrends_metrics(occupations):
    if LOCALTRENDS_CACHE.exists():
        cached = pd.read_csv(LOCALTRENDS_CACHE)
        if set(cached['soc_code']) == set(occupations['soc_code']):
            return cached

    growth_pattern = re.compile(
        r'Projected growth <small class="d-sm-block">\(2024-2034\)</small></dt>\s*<dd[^>]*>.*?([-\d]+)%',
        re.S,
    )
    openings_pattern = re.compile(
        r'Projected annual <span class="d-sm-block">job openings</span> <small class="d-sm-block">\(2024-2034\)</small></dt>\s*<dd[^>]*>([\d,]+)',
        re.S,
    )
    rows = []
    for soc_code, occupation_title in occupations[['soc_code', 'occupation_title']].itertuples(index=False):
        url = f'https://www.onetonline.org/link/localtrends/{soc_code}.00'
        request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        html = urllib.request.urlopen(request, timeout=30).read().decode('utf-8', 'ignore')
        growth_match = growth_pattern.search(html)
        openings_match = openings_pattern.search(html)
        rows.append(
            {
                'soc_code': soc_code,
                'source_url': url,
                'projected_growth_pct': int(growth_match.group(1)),
                'annual_openings': int(openings_match.group(1).replace(',', '')) if openings_match else pd.NA,
            }
        )
    localtrends = pd.DataFrame(rows)
    localtrends.to_csv(LOCALTRENDS_CACHE, index=False)
    return localtrends


In [ ]:
aioe = pd.read_excel(AIOE_XLSX, sheet_name='Appendix A')
aioe['soc_code'] = normalize_soc(aioe['SOC Code'])
aioe = aioe.rename(columns={'Occupation Title': 'aioe_occupation_title', 'AIOE': 'ai_exposure'})[
    ['soc_code', 'aioe_occupation_title', 'ai_exposure']
]

with zipfile.ZipFile(ONET_ZIP) as onet_zip:
    occupation_data = read_onet_excel_from_zip(onet_zip, 'Occupation Data.xlsx')
    job_zones = read_onet_excel_from_zip(onet_zip, 'Job Zones.xlsx')
    skills = read_onet_excel_from_zip(onet_zip, 'Skills.xlsx')
    knowledge = read_onet_excel_from_zip(onet_zip, 'Knowledge.xlsx')
    work_activities = read_onet_excel_from_zip(onet_zip, 'Work Activities.xlsx')

for df, col in [
    (occupation_data, 'O*NET-SOC Code'),
    (job_zones, 'O*NET-SOC Code'),
    (skills, 'O*NET-SOC Code'),
    (knowledge, 'O*NET-SOC Code'),
    (work_activities, 'O*NET-SOC Code'),
]:
    df['soc_code'] = normalize_soc(df[col])

family = occupation_data[occupation_data['soc_code'].str.startswith('13-', na=False)].copy()
family = (
    family.sort_values(['soc_code', 'O*NET-SOC Code'])
    .drop_duplicates('soc_code')
    .rename(columns={'Title': 'occupation_title'})[['soc_code', 'occupation_title']]
    .reset_index(drop=True)
)

merged = family.merge(aioe, on='soc_code', how='left')
merged = merged.merge(job_zones.groupby('soc_code')['Job Zone'].mean().rename('job_zone'), on='soc_code', how='left')

composites = [
    aggregate_scale_average(skills, ['Critical Thinking', 'Complex Problem Solving', 'Judgment and Decision Making', 'Active Learning'], 'LV', 'judgment_skills'),
    aggregate_scale_average(skills, ['Mathematics'], 'LV', 'math_skill'),
    aggregate_scale_average(knowledge, ['Economics and Accounting', 'Mathematics', 'Customer and Personal Service'], 'LV', 'finance_knowledge'),
    aggregate_scale_average(work_activities, ['Processing Information', 'Documenting/Recording Information', 'Updating and Using Relevant Knowledge'], 'LV', 'process_activities'),
    aggregate_scale_average(work_activities, ['Analyzing Data or Information', 'Interpreting the Meaning of Information for Others', 'Providing Consultation and Advice to Others'], 'LV', 'advice_analysis'),
]
for series in composites:
    merged = merged.merge(series, on='soc_code', how='left')

localtrends = fetch_localtrends_metrics(family)
merged = merged.merge(localtrends, on='soc_code', how='left')
merged['occupation_title'] = merged['occupation_title'].fillna(merged['aioe_occupation_title'])
merged['task_balance'] = merged['advice_analysis'] - merged['process_activities']
for col in FEATURE_COLS:
    merged[col] = merged[col].fillna(merged[col].mean())

merged[['soc_code', 'occupation_title', 'projected_growth_pct', 'annual_openings'] + FEATURE_COLS].head()


In [ ]:
ml_dataset = merged[merged['projected_growth_pct'].notna()].copy().reset_index(drop=True)
X = ml_dataset[FEATURE_COLS].to_numpy(dtype=float)
y = ml_dataset['projected_growth_pct'].to_numpy(dtype=float)

feature_means = X.mean(axis=0)
feature_stds = np.where(X.std(axis=0) == 0, 1, X.std(axis=0))
X_standardized = (X - feature_means) / feature_stds

loocv_predictions = []
for i in range(len(ml_dataset)):
    mask = np.ones(len(ml_dataset), dtype=bool)
    mask[i] = False
    beta = fit_linear_regression(X_standardized[mask], y[mask])
    loocv_predictions.append(predict_linear_regression(beta, X_standardized[~mask])[0])

ml_dataset['predicted_projected_growth'] = loocv_predictions
ml_dataset['prediction_error'] = ml_dataset['projected_growth_pct'] - ml_dataset['predicted_projected_growth']

final_beta = fit_linear_regression(X_standardized, y)
model_coefficients = pd.DataFrame({'feature': FEATURE_COLS, 'coefficient': final_beta[1:]})
model_metrics = pd.DataFrame(
    [
        {'metric': 'loocv_r2', 'value': r2_score(y, ml_dataset['predicted_projected_growth'])},
        {'metric': 'loocv_mae', 'value': mae(y, ml_dataset['predicted_projected_growth'])},
        {'metric': 'n_business_financial_occupations', 'value': len(ml_dataset)},
    ]
)

model_metrics


In [ ]:
finance_subset = ml_dataset[ml_dataset['soc_code'].isin(FINANCE_GROUPS.keys())].copy()
finance_subset['job_group'] = finance_subset['soc_code'].map(FINANCE_GROUPS)
finance_subset = finance_subset.sort_values(['projected_growth_pct', 'predicted_projected_growth'], ascending=[False, False]).reset_index(drop=True)

sample_1 = finance_subset.loc[finance_subset['job_group'] == 'judgment-intensive', 'projected_growth_pct']
sample_2 = finance_subset.loc[finance_subset['job_group'] == 'process-driven', 'projected_growth_pct']
t_test = welch_t_test(sample_1, sample_2, alternative='greater')

alpha = 0.05
decision = 'reject H0' if t_test['p_value'] < alpha else 'fail to reject H0'
t_test_results = pd.DataFrame(
    [
        {
            'response_variable': 'projected_growth_pct',
            'group_1': 'judgment-intensive',
            'group_2': 'process-driven',
            'alternative_hypothesis': 'mu_1 > mu_2',
            'alpha': alpha,
            'n_1': t_test['n_1'],
            'n_2': t_test['n_2'],
            'mean_1': t_test['mean_1'],
            'mean_2': t_test['mean_2'],
            't_statistic': t_test['t_statistic'],
            'degrees_of_freedom': t_test['degrees_of_freedom'],
            'p_value': t_test['p_value'],
            'decision': decision,
        }
    ]
)

finance_subset[['occupation_title', 'job_group', 'projected_growth_pct', 'predicted_projected_growth']]
